In [3]:
import numpy as np

def calculate_ahp_weights(matrix):
    """
    Calculates AHP weights and Consistency Ratio from a pairwise comparison matrix.

    Args:
        matrix (list of lists): The pairwise comparison matrix.

    Returns:
        tuple: A tuple containing:
            - weights (numpy.ndarray): The calculated weights for each factor.
            - cr (float): The Consistency Ratio.
            - ci (float): The Consistency Index.
            - ri (float): The Random Index used.
    """
    n = len(matrix)
    if n == 0:
        print("Error: Matrix is empty.")
        return None, None, None, None

    # Convert the list of lists to a NumPy array for easier calculations
    matrix_np = np.array(matrix, dtype=float)

    # Check for square matrix
    if matrix_np.shape!= matrix_np.shape[1]:
        print("Error: Matrix is not square.")
        return None, None, None, None

    # Check for positive values and reciprocals (basic check)
    if not np.all(matrix_np > 0):
        print("Error: All matrix elements must be positive.")
        return None, None, None, None
    
    # Check for consistency of reciprocals (a_ij * a_ji = 1)
    # This is a basic check; full AHP requires this by construction.
    for i in range(n):
        for j in range(n):
            if i!= j and abs(matrix_np[i, j] * matrix_np[j, i] - 1) > 1e-9:
                print(f"Warning: Matrix element at ({i},{j}) and ({j},{i}) are not perfect reciprocals.")


    # Step 1: Normalize the matrix (sum of each column to 1)
    # This is done by dividing each element by the sum of its column.
    normalized_matrix = matrix_np / matrix_np.sum(axis=0)

    # Step 2: Calculate the row averages (weights)
    # The average of each row in the normalized matrix gives the priority vector (weights).
    weights = normalized_matrix.mean(axis=1)
    
    # Normalize weights to sum to 1 (optional, but good practice for priorities)
    weights = weights / weights.sum()

    # Step 3: Calculate the Consistency Index (CI)
    # Multiply the original matrix by the weights vector
    weighted_sum_vector = np.dot(matrix_np, weights)

    # Divide the weighted sum vector by the weights vector to get lambda_max
    lambda_max_vector = weighted_sum_vector / weights
    lambda_max = lambda_max_vector.mean() # Average of these values is lambda_max

    ci = (lambda_max - n) / (n - 1)

    # Step 4: Calculate the Consistency Ratio (CR)
    # Random Index (RI) values for n=1 to n=15 (from Saaty's table)
    # Source: Saaty, T. L. (1980). The Analytic Hierarchy Process. McGraw-Hill.
    ri_values = {
        1: 0.00, 2: 0.00, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41,
        9: 1.45, 10: 1.49, 11: 1.51, 12: 1.48, 13: 1.56, 14: 1.57, 15: 1.59
    }

    if n not in ri_values:
        print(f"Warning: No Random Index (RI) value available for n={n}. CR cannot be calculated.")
        ri = None
        cr = None
    else:
        ri = ri_values[n]
        if ri == 0: # For n=1 or n=2, CI is always 0, so CR is 0.
            cr = 0.0
        else:
            cr = ci / ri

    return weights, cr, ci, ri

# --- How to Use This Code ---

# 1. Define your factors in the correct order. This order must match the rows/columns of your Excel matrix.
my_factors = [
    "Fault", "Aspect", "Slope", "Topographic Water Index", "Geology", "DEM", "LULC",
    "Rainfall", "Distance to Rail Crossing", "Distance Road", "Distance to Bridge",
    "Distance to River", "Distance to Waterbody"
]

# 2. Create your 13x13 pairwise comparison matrix (my_pairwise_matrix).
#    Each row in this list represents a row from your Excel matrix.
#    - Diagonal elements (comparing a factor to itself) must be 1.
#    - If Factor A vs. Factor B is 'X', then Factor B vs. Factor A must be '1/X'.
#    - Use Saaty's scale: 1, 3, 5, 7, 9 for importance, and 1/3, 1/5, 1/7, 1/9 for less importance.
#    - Intermediate values (2, 4, 6, 8) can also be used.

#    *** REPLACE THE DUMMY VALUES BELOW WITH YOUR ACTUAL 13x13 MATRIX FROM EXCEL ***
#    Each inner list corresponds to a row in your Excel sheet.
#    Example: The first inner list is the 'Fault' row, comparing Fault to all other 12 factors.
my_pairwise_matrix = [
    [1, 1/3, 1/5, 1/7, 1/9, 1/3, 1/5, 1/7, 1/9, 1/3, 1/5, 1/7, 1/9], # Fault vs. all others (dummy values)
    [3, 1, 1/3, 1/5, 1/7, 1/9, 1/3, 1/5, 1/7, 1/9, 1/3, 1/5, 1/7],  # Aspect vs. all others (dummy values)
    [5, 3, 1, 1/3, 1/5, 1/7, 1/9, 1/3, 1/5, 1/7, 1/9, 1/3, 1/5],   # Slope vs. all others (dummy values)
    [7, 5, 3, 1, 1/3, 1/5, 1/7, 1/9, 1/3, 1/5, 1/7, 1/9, 1/3],    # Topographic Water Index vs. all others (dummy values)
    [9, 7, 5, 3, 1, 1/3, 1/5, 1/7, 1/9, 1/3, 1/5, 1/7, 1/9],     # Geology vs. all others (dummy values)
    [3, 9, 7, 5, 3, 1, 1/3, 1/5, 1/7, 1/9, 1/3, 1/5, 1/7],      # DEM vs. all others (dummy values)
    [5, 3, 9, 7, 5, 3, 1, 1/3, 1/5, 1/7, 1/9, 1/3, 1/5],       # LULC vs. all others (dummy values)
    [7, 5, 3, 9, 7, 5, 3, 1, 1/3, 1/5, 1/7, 1/9, 1/3],        # Rainfall vs. all others (dummy values)
    [9, 7, 5, 3, 9, 7, 5, 3, 1, 1/3, 1/5, 1/7, 1/9],         # Distance to Rail Crossing vs. all others (dummy values)
    [3, 9, 7, 5, 3, 9, 7, 5, 3, 1, 1/3, 1/5, 1/7],          # Distance Road vs. all others (dummy values)
    [5, 3, 9, 7, 5, 3, 9, 7, 5, 3, 1, 1/3, 1/5],           # Distance to Bridge vs. all others (dummy values)
    [7, 5, 3, 9, 7, 5, 3, 9, 7, 5, 3, 1, 1/3],            # Distance to River vs. all others (dummy values)
    [9, 7, 5, 3, 9, 7, 5, 3, 9, 7, 5, 3, 1]               # Distance to Waterbody vs. all others (dummy values)
]


# 3. Run the calculation
if len(my_pairwise_matrix) == len(my_factors):
    weights, cr, ci, ri = calculate_ahp_weights(my_pairwise_matrix)

    if weights is not None:
        print("\n--- AHP Results ---")
        print("Calculated Weights (Priorities):")
        for i, weight in enumerate(weights):
            print(f"{my_factors[i]}: {weight:.4f}")

        print(f"\nConsistency Index (CI): {ci:.4f}")
        print(f"Random Index (RI) for n={len(my_factors)}: {ri:.2f}")
        print(f"Consistency Ratio (CR): {cr:.4f}")

        if cr is not None:
            if cr < 0.10:
                print("\nConsistency is acceptable (CR < 0.10).")
            else:
                print("\nConsistency is NOT acceptable (CR >= 0.10). Please review your pairwise comparisons for inconsistencies.")
    else:
        print("\nCould not calculate AHP weights due to an error in the matrix. Please check the matrix format and values.")
else:
    print("Error: The number of factors defined does not match the size of the provided matrix.")

Error: Matrix is not square.

Could not calculate AHP weights due to an error in the matrix. Please check the matrix format and values.


In [4]:
# Diagnostic: Check matrix dimensions
print(f"Number of factors: {len(my_factors)}")
print(f"Number of matrix rows: {len(my_pairwise_matrix)}")
print("Number of elements in each row:")
for i, row in enumerate(my_pairwise_matrix):
    print(f"Row {i+1}: {len(row)} elements")

# Convert to numpy to see the exact shape issue
import numpy as np
try:
    matrix_np = np.array(my_pairwise_matrix, dtype=float)
    print(f"\nMatrix shape: {matrix_np.shape}")
except Exception as e:
    print(f"\nError converting to numpy array: {e}")

Number of factors: 13
Number of matrix rows: 13
Number of elements in each row:
Row 1: 13 elements
Row 2: 13 elements
Row 3: 13 elements
Row 4: 13 elements
Row 5: 13 elements
Row 6: 13 elements
Row 7: 13 elements
Row 8: 13 elements
Row 9: 13 elements
Row 10: 13 elements
Row 11: 13 elements
Row 12: 13 elements
Row 13: 13 elements

Matrix shape: (13, 13)
